# Lightweight POD / rSVD / DEIM GIF experiments

This notebook is a compact version of the original PDE/POD/rSVD/DEIM experiments.

It generates four GIFs:

1. `gif_outputs/allen_cahn_pod.gif`
2. `gif_outputs/allen_cahn_deim.gif`
3. `gif_outputs/elliptic_pod.gif`
4. `gif_outputs/elliptic_deim.gif`

Each GIF has one row and three columns:

- **Full-order PDE**
- **Reduced model with deterministic SVD**
- **Reduced model with randomized SVD**

For the reduced columns, the title reports the relative reconstruction error with respect to the full-order reference.

The defaults are intentionally small so that the whole notebook can be run quickly on a laptop or in Colab.

## 1. Imports and global parameters

In [ ]:
import os
import time
from dataclasses import dataclass

import numpy as np
import scipy.linalg as la
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from matplotlib.animation import PillowWriter

np.random.seed(42)

# Output folder for the GIFs
GIF_DIR = "gif_outputs"
os.makedirs(GIF_DIR, exist_ok=True)

# Small defaults: increase these only if you want smoother or higher-resolution GIFs.
ALLEN_N = 28
ALLEN_T = 0.25
ALLEN_DT = 0.005
ALLEN_SNAPSHOT_STRIDE = 2

ELL_N = 18
ELL_TRAIN_GRID = 5
ELL_PATH_FRAMES = 14

# Low ranks requested by the prompt.
R_STATE = 5
M_DEIM = 5

# rSVD settings.
RSVD_OVERSAMPLING = 4
RSVD_POWER_ITER = 1

# Animation settings.
GIF_FPS = 6
CMAP = "viridis"

## 2. Shared numerical utilities

In [ ]:
def make_laplacian_2d(n):
    """
    2D finite-difference Laplacian on (0,1)^2 with homogeneous Dirichlet boundary conditions.

    The unknowns are the n*n interior grid points.
    """
    h = 1.0 / (n + 1)
    x = np.linspace(h, 1.0 - h, n)
    X, Y = np.meshgrid(x, x, indexing="ij")

    e = np.ones(n)
    T = sp.diags([e, -2.0 * e, e], [-1, 0, 1], shape=(n, n), format="csr") / h**2
    I = sp.eye(n, format="csr")
    L = sp.kron(I, T) + sp.kron(T, I)
    return L.tocsr(), h, X, Y


def relative_fro_error(A, Ahat):
    den = la.norm(A, ord="fro")
    if den == 0:
        return la.norm(A - Ahat, ord="fro")
    return la.norm(A - Ahat, ord="fro") / den


def relative_l2_error(u, uhat):
    den = la.norm(u)
    if den == 0:
        return la.norm(u - uhat)
    return la.norm(u - uhat) / den


def deterministic_svd_basis(S, r):
    """
    POD basis from deterministic SVD.
    """
    U, s, Vt = la.svd(S, full_matrices=False)
    r_eff = min(r, U.shape[1])
    return U[:, :r_eff], s[:r_eff]


def randomized_svd_basis(S, r, oversampling=5, n_iter=1):
    """
    Simple randomized SVD implementation.

    Given S, it approximates the dominant left singular vectors.
    This avoids relying on scikit-learn.
    """
    n_rows, n_cols = S.shape
    l = min(r + oversampling, min(n_rows, n_cols))

    Omega = np.random.randn(n_cols, l)
    Y = S @ Omega

    for _ in range(n_iter):
        Y = S @ (S.T @ Y)

    Q, _ = la.qr(Y, mode="economic")
    B = Q.T @ S
    Uh, s, Vt = la.svd(B, full_matrices=False)
    U = Q @ Uh

    r_eff = min(r, U.shape[1])
    return U[:, :r_eff], s[:r_eff]


def deim_indices(U):
    """
    Greedy DEIM interpolation point selection.

    Parameters
    ----------
    U : ndarray, shape (N, m)
        Nonlinear basis.

    Returns
    -------
    indices : ndarray, shape (m,)
        Selected interpolation indices.
    """
    m = U.shape[1]
    indices = []
    indices.append(int(np.argmax(np.abs(U[:, 0]))))

    for j in range(1, m):
        U_prev = U[:, :j]
        p_prev = np.array(indices)
        c = la.solve(U_prev[p_prev, :], U[p_prev, j], assume_a="gen")
        residual = U[:, j] - U_prev @ c
        indices.append(int(np.argmax(np.abs(residual))))

    return np.array(indices, dtype=int)


def deim_precompute(U):
    """
    Precompute objects for DEIM approximation:

        f(y) ≈ U @ inv(U[P,:]) @ f(y)[P]
    """
    idx = deim_indices(U)
    inv_UP = la.inv(U[idx, :])
    return idx, inv_UP


def deim_approx_from_sampled_values(U, idx, inv_UP, f_idx):
    return U @ (inv_UP @ f_idx)


def build_gif_three_columns(S_full, S_svd, S_rsvd, n, times_or_labels, filename, suptitle,
                            left_title="Full-order PDE",
                            middle_title="Reduced with SVD",
                            right_title="Reduced with rSVD",
                            middle_error=None,
                            right_error=None,
                            fps=GIF_FPS,
                            cmap=CMAP):
    """
    Build a 1x3 GIF comparing full-order, SVD-reduced and rSVD-reduced fields.

    S_* matrices have shape (n*n, n_frames).
    """
    n_frames = S_full.shape[1]
    vmin = min(S_full.min(), S_svd.min(), S_rsvd.min())
    vmax = max(S_full.max(), S_svd.max(), S_rsvd.max())

    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6), constrained_layout=True)

    titles = [
        left_title,
        middle_title if middle_error is None else f"{middle_title}\nrel. error = {middle_error:.2e}",
        right_title if right_error is None else f"{right_title}\nrel. error = {right_error:.2e}",
    ]

    images = []
    for ax, S, title in zip(axes, [S_full, S_svd, S_rsvd], titles):
        im = ax.imshow(S[:, 0].reshape(n, n), origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")
        images.append(im)

    frame_label = fig.text(0.5, 0.02, "", ha="center", fontsize=11)
    fig.suptitle(suptitle, fontweight="bold")

    writer = PillowWriter(fps=fps)
    out_path = os.path.join(GIF_DIR, filename)

    with writer.saving(fig, out_path, dpi=85):
        for k in range(n_frames):
            for im, S in zip(images, [S_full, S_svd, S_rsvd]):
                im.set_data(S[:, k].reshape(n, n))

            label = times_or_labels[k]
            if isinstance(label, str):
                frame_label.set_text(label)
            else:
                frame_label.set_text(f"t = {label:.3f}")

            writer.grab_frame()

    plt.close(fig)
    print(f"Saved: {out_path}")
    return out_path

## 3. Allen--Cahn full-order model

We solve

\[
y_t = \alpha \Delta y + \mu(y-y^3)
\]

with a semi-implicit finite-difference method:

\[
(I-\Delta t \alpha L)y^{n+1}
=
y^n + \Delta t \mu(y^n-(y^n)^3).
\]

In [ ]:
@dataclass
class AllenCahnConfig:
    n: int = ALLEN_N
    alpha: float = 0.1
    mu: float = 11.0
    T: float = ALLEN_T
    dt: float = ALLEN_DT
    snapshot_stride: int = ALLEN_SNAPSHOT_STRIDE


def allen_cahn_initial_condition(X, Y):
    return 0.1 * np.sin(np.pi * X) * np.sin(np.pi * Y)


def allen_cahn_nonlinearity(y, mu):
    return mu * (y - y**3)


def solve_allen_cahn_full(config):
    L, h, X, Y = make_laplacian_2d(config.n)
    N = config.n**2
    I = sp.eye(N, format="csr")

    A = I - config.dt * config.alpha * L
    solver = spla.factorized(A.tocsc())

    y = allen_cahn_initial_condition(X, Y).reshape(-1)

    n_steps = int(round(config.T / config.dt))
    snapshots = []
    nonlinear_snapshots = []
    times = []

    t0 = time.perf_counter()
    for step in range(n_steps + 1):
        if step % config.snapshot_stride == 0:
            snapshots.append(y.copy())
            nonlinear_snapshots.append(allen_cahn_nonlinearity(y, config.mu))
            times.append(step * config.dt)

        if step < n_steps:
            rhs = y + config.dt * allen_cahn_nonlinearity(y, config.mu)
            y = solver(rhs)

    elapsed = time.perf_counter() - t0

    return {
        "S": np.column_stack(snapshots),
        "F": np.column_stack(nonlinear_snapshots),
        "times": np.array(times),
        "L": L,
        "X": X,
        "Y": Y,
        "elapsed": elapsed,
    }


ac_config = AllenCahnConfig()
ac_data = solve_allen_cahn_full(ac_config)
S_ac = ac_data["S"]
F_ac = ac_data["F"]
times_ac = ac_data["times"]

print("Allen-Cahn snapshots:", S_ac.shape)
print(f"Full-order solve time: {ac_data['elapsed']:.3f} s")

## 4. Allen--Cahn reduced models: POD and rSVD-POD

In [ ]:
def simulate_allen_cahn_pod_rom(V, config, L, y0):
    """
    Semi-implicit POD Galerkin ROM without DEIM.

    y ≈ V a
    """
    dt = config.dt
    alpha = config.alpha

    L_V = L @ V
    A_r = np.eye(V.shape[1]) - dt * alpha * (V.T @ L_V)
    lu_r, piv_r = la.lu_factor(A_r)

    a = V.T @ y0

    n_steps = int(round(config.T / config.dt))
    snapshots = []
    for step in range(n_steps + 1):
        if step % config.snapshot_stride == 0:
            snapshots.append(V @ a)

        if step < n_steps:
            y_rom = V @ a
            rhs = a + dt * (V.T @ allen_cahn_nonlinearity(y_rom, config.mu))
            a = la.lu_solve((lu_r, piv_r), rhs)

    return np.column_stack(snapshots)


# State bases
V_ac_svd, _ = deterministic_svd_basis(S_ac, R_STATE)
V_ac_rsvd, _ = randomized_svd_basis(S_ac, R_STATE, RSVD_OVERSAMPLING, RSVD_POWER_ITER)

# Initial condition is the first full-order snapshot.
y0_ac = S_ac[:, 0]

S_ac_pod_svd = simulate_allen_cahn_pod_rom(V_ac_svd, ac_config, ac_data["L"], y0_ac)
S_ac_pod_rsvd = simulate_allen_cahn_pod_rom(V_ac_rsvd, ac_config, ac_data["L"], y0_ac)

err_ac_pod_svd = relative_fro_error(S_ac, S_ac_pod_svd)
err_ac_pod_rsvd = relative_fro_error(S_ac, S_ac_pod_rsvd)

print(f"Allen-Cahn POD-SVD ROM error:  {err_ac_pod_svd:.3e}")
print(f"Allen-Cahn POD-rSVD ROM error: {err_ac_pod_rsvd:.3e}")

## 5. Allen--Cahn reduced models: POD-DEIM and rSVD-DEIM

In [ ]:
def simulate_allen_cahn_pod_deim_rom(V, U_f, config, L, y0):
    """
    Semi-implicit POD-DEIM ROM.

    Nonlinear term:
        f(Va) ≈ U_f inv(U_f[P,:]) f(Va)[P]
    """
    dt = config.dt
    alpha = config.alpha
    r = V.shape[1]

    idx, inv_UP = deim_precompute(U_f)

    L_V = L @ V
    A_r = np.eye(r) - dt * alpha * (V.T @ L_V)
    lu_r, piv_r = la.lu_factor(A_r)

    # Projection of the DEIM approximation onto the state basis.
    B = V.T @ U_f @ inv_UP

    a = V.T @ y0

    n_steps = int(round(config.T / config.dt))
    snapshots = []

    for step in range(n_steps + 1):
        if step % config.snapshot_stride == 0:
            snapshots.append(V @ a)

        if step < n_steps:
            y_sampled = V[idx, :] @ a
            f_sampled = allen_cahn_nonlinearity(y_sampled, config.mu)
            rhs = a + dt * (B @ f_sampled)
            a = la.lu_solve((lu_r, piv_r), rhs)

    return np.column_stack(snapshots)


# Nonlinear bases for DEIM
U_ac_f_svd, _ = deterministic_svd_basis(F_ac, M_DEIM)
U_ac_f_rsvd, _ = randomized_svd_basis(F_ac, M_DEIM, RSVD_OVERSAMPLING, RSVD_POWER_ITER)

S_ac_deim_svd = simulate_allen_cahn_pod_deim_rom(V_ac_svd, U_ac_f_svd, ac_config, ac_data["L"], y0_ac)
S_ac_deim_rsvd = simulate_allen_cahn_pod_deim_rom(V_ac_rsvd, U_ac_f_rsvd, ac_config, ac_data["L"], y0_ac)

err_ac_deim_svd = relative_fro_error(S_ac, S_ac_deim_svd)
err_ac_deim_rsvd = relative_fro_error(S_ac, S_ac_deim_rsvd)

print(f"Allen-Cahn POD-DEIM SVD ROM error:  {err_ac_deim_svd:.3e}")
print(f"Allen-Cahn POD-DEIM rSVD ROM error: {err_ac_deim_rsvd:.3e}")

## 6. Elliptic full-order model

In [ ]:
@dataclass
class EllipticConfig:
    n: int = ELL_N
    newton_tol: float = 1e-8
    newton_max_iter: int = 20


def elliptic_rhs(X, Y):
    return 100.0 * np.sin(2 * np.pi * X) * np.sin(2 * np.pi * Y)


def nonlinear_s(u, mu1, mu2):
    z = np.clip(mu2 * u, -50, 50)
    return (mu1 / mu2) * (np.exp(z) - 1.0)


def nonlinear_s_prime(u, mu1, mu2):
    z = np.clip(mu2 * u, -50, 50)
    return mu1 * np.exp(z)


def solve_elliptic_full(mu1, mu2, config, initial_guess=None):
    """
    Solve:
        -Delta u + s(u; mu) = rhs
    using Newton's method.
    """
    L, h, X, Y = make_laplacian_2d(config.n)
    N = config.n**2
    rhs = elliptic_rhs(X, Y).reshape(-1)

    if initial_guess is None:
        u = np.zeros(N)
    else:
        u = initial_guess.copy()

    for it in range(config.newton_max_iter):
        s_val = nonlinear_s(u, mu1, mu2)
        residual = -L @ u + s_val - rhs
        res_norm = la.norm(residual)

        if res_norm < config.newton_tol:
            return {"u": u, "converged": True, "iterations": it, "L": L, "X": X, "Y": Y}

        sp_val = nonlinear_s_prime(u, mu1, mu2)
        J = -L + sp.diags(sp_val, 0, format="csr")
        delta = spla.spsolve(J, -residual)
        u = u + delta

        if la.norm(delta) < config.newton_tol * max(1.0, la.norm(u)):
            return {"u": u, "converged": True, "iterations": it + 1, "L": L, "X": X, "Y": Y}

    return {"u": u, "converged": False, "iterations": config.newton_max_iter, "L": L, "X": X, "Y": Y}


def generate_elliptic_training_snapshots(config, grid_size=ELL_TRAIN_GRID):
    mu1_values = np.linspace(0.01, 10.0, grid_size)
    mu2_values = np.linspace(0.01, 10.0, grid_size)

    snapshots = []
    nonlinear_snapshots = []
    params = []

    previous_u = None
    L_ref = X_ref = Y_ref = None

    t0 = time.perf_counter()
    for mu1 in mu1_values:
        for mu2 in mu2_values:
            sol = solve_elliptic_full(mu1, mu2, config, initial_guess=previous_u)
            if not sol["converged"]:
                print(f"Warning: Newton did not fully converge for mu=({mu1:.2f}, {mu2:.2f})")

            u = sol["u"]
            previous_u = u

            snapshots.append(u)
            nonlinear_snapshots.append(nonlinear_s(u, mu1, mu2))
            params.append((mu1, mu2))

            L_ref = sol["L"]
            X_ref = sol["X"]
            Y_ref = sol["Y"]

    elapsed = time.perf_counter() - t0

    return {
        "S": np.column_stack(snapshots),
        "F": np.column_stack(nonlinear_snapshots),
        "params": params,
        "L": L_ref,
        "X": X_ref,
        "Y": Y_ref,
        "elapsed": elapsed,
    }


ell_config = EllipticConfig()
ell_train = generate_elliptic_training_snapshots(ell_config, ELL_TRAIN_GRID)

S_ell_train = ell_train["S"]
F_ell_train = ell_train["F"]

print("Elliptic training snapshots:", S_ell_train.shape)
print(f"Training full-order solve time: {ell_train['elapsed']:.3f} s")

## 7. Elliptic parameter path

The elliptic PDE is steady-state, so there is no physical time evolution.  
For the GIF, we create frames by moving along a parameter path:

\[
(\mu_1,\mu_2): (0.01,0.01) \to (10,10).
\]

This produces a meaningful sequence of changing full-order solutions.

In [ ]:
def generate_elliptic_path(config, n_frames=ELL_PATH_FRAMES):
    mu1_path = np.linspace(0.01, 10.0, n_frames)
    mu2_path = np.linspace(0.01, 10.0, n_frames)

    snapshots = []
    nonlinear_snapshots = []
    params = []

    previous_u = None
    L_ref = X_ref = Y_ref = None

    for mu1, mu2 in zip(mu1_path, mu2_path):
        sol = solve_elliptic_full(mu1, mu2, config, initial_guess=previous_u)
        if not sol["converged"]:
            print(f"Warning: Newton did not fully converge on path for mu=({mu1:.2f}, {mu2:.2f})")

        u = sol["u"]
        previous_u = u

        snapshots.append(u)
        nonlinear_snapshots.append(nonlinear_s(u, mu1, mu2))
        params.append((mu1, mu2))

        L_ref = sol["L"]
        X_ref = sol["X"]
        Y_ref = sol["Y"]

    labels = [rf"$\mu_1={mu1:.2f}$, $\mu_2={mu2:.2f}$" for mu1, mu2 in params]

    return {
        "S": np.column_stack(snapshots),
        "F": np.column_stack(nonlinear_snapshots),
        "params": params,
        "labels": labels,
        "L": L_ref,
        "X": X_ref,
        "Y": Y_ref,
    }


ell_path = generate_elliptic_path(ell_config, ELL_PATH_FRAMES)
S_ell_path = ell_path["S"]
F_ell_path = ell_path["F"]
labels_ell = ell_path["labels"]

print("Elliptic path snapshots:", S_ell_path.shape)

## 8. Elliptic reduced models: POD and rSVD-POD

In [ ]:
# State bases from training snapshots
V_ell_svd, _ = deterministic_svd_basis(S_ell_train, R_STATE)
V_ell_rsvd, _ = randomized_svd_basis(S_ell_train, R_STATE, RSVD_OVERSAMPLING, RSVD_POWER_ITER)

# Reuse a fixed grid and RHS in the reduced solvers.
_, _, X_ell, Y_ell = make_laplacian_2d(ell_config.n)
rhs_ell = elliptic_rhs(X_ell, Y_ell).reshape(-1)

def solve_elliptic_pod_rom(mu1, mu2, V, config, L, rhs, initial_a=None):
    """
    Reduced Newton solve without DEIM:

        V^T[-L V a + s(Va;mu) - rhs] = 0
    """
    r = V.shape[1]
    if initial_a is None:
        a = np.zeros(r)
    else:
        a = initial_a.copy()

    LV = L @ V
    K_r = V.T @ (-LV)
    rhs_r = V.T @ rhs

    for it in range(config.newton_max_iter):
        u = V @ a
        s_val = nonlinear_s(u, mu1, mu2)
        residual = K_r @ a + V.T @ s_val - rhs_r

        if la.norm(residual) < config.newton_tol:
            return a, True

        sp_val = nonlinear_s_prime(u, mu1, mu2)
        J_r = K_r + V.T @ (sp_val[:, None] * V)
        delta = la.solve(J_r, -residual, assume_a="gen")
        a = a + delta

        if la.norm(delta) < config.newton_tol * max(1.0, la.norm(a)):
            return a, True

    return a, False


def solve_elliptic_path_pod_rom(params, V, config, L, rhs):
    snapshots = []
    a_prev = None

    for mu1, mu2 in params:
        a, converged = solve_elliptic_pod_rom(mu1, mu2, V, config, L, rhs, initial_a=a_prev)
        if not converged:
            print(f"Warning: POD ROM Newton did not fully converge for mu=({mu1:.2f}, {mu2:.2f})")
        a_prev = a
        snapshots.append(V @ a)

    return np.column_stack(snapshots)


S_ell_pod_svd = solve_elliptic_path_pod_rom(ell_path["params"], V_ell_svd, ell_config, ell_path["L"], rhs_ell)
S_ell_pod_rsvd = solve_elliptic_path_pod_rom(ell_path["params"], V_ell_rsvd, ell_config, ell_path["L"], rhs_ell)

err_ell_pod_svd = relative_fro_error(S_ell_path, S_ell_pod_svd)
err_ell_pod_rsvd = relative_fro_error(S_ell_path, S_ell_pod_rsvd)

print(f"Elliptic POD-SVD ROM error:  {err_ell_pod_svd:.3e}")
print(f"Elliptic POD-rSVD ROM error: {err_ell_pod_rsvd:.3e}")

## 9. Elliptic reduced models: POD-DEIM and rSVD-DEIM

In [ ]:
def solve_elliptic_pod_deim_rom(mu1, mu2, V, U_f, config, L, rhs, initial_a=None):
    """
    Reduced Newton solve with DEIM approximation of the nonlinear term:

        s(Va;mu) ≈ U_f inv(U_f[P,:]) s((Va)[P];mu)

    Reduced residual:
        V^T[-LVa + DEIM_s(Va) - rhs] = 0
    """
    idx, inv_UP = deim_precompute(U_f)

    r = V.shape[1]
    if initial_a is None:
        a = np.zeros(r)
    else:
        a = initial_a.copy()

    LV = L @ V
    K_r = V.T @ (-LV)
    rhs_r = V.T @ rhs

    # B maps sampled nonlinear values into the reduced residual.
    B = V.T @ U_f @ inv_UP
    V_sample = V[idx, :]

    for it in range(config.newton_max_iter):
        u_sample = V_sample @ a
        s_sample = nonlinear_s(u_sample, mu1, mu2)
        residual = K_r @ a + B @ s_sample - rhs_r

        if la.norm(residual) < config.newton_tol:
            return a, True

        sp_sample = nonlinear_s_prime(u_sample, mu1, mu2)
        J_r = K_r + B @ (sp_sample[:, None] * V_sample)

        delta = la.solve(J_r, -residual, assume_a="gen")
        a = a + delta

        if la.norm(delta) < config.newton_tol * max(1.0, la.norm(a)):
            return a, True

    return a, False


def solve_elliptic_path_pod_deim_rom(params, V, U_f, config, L, rhs):
    snapshots = []
    a_prev = None

    for mu1, mu2 in params:
        a, converged = solve_elliptic_pod_deim_rom(mu1, mu2, V, U_f, config, L, rhs, initial_a=a_prev)
        if not converged:
            print(f"Warning: POD-DEIM ROM Newton did not fully converge for mu=({mu1:.2f}, {mu2:.2f})")
        a_prev = a
        snapshots.append(V @ a)

    return np.column_stack(snapshots)


U_ell_f_svd, _ = deterministic_svd_basis(F_ell_train, M_DEIM)
U_ell_f_rsvd, _ = randomized_svd_basis(F_ell_train, M_DEIM, RSVD_OVERSAMPLING, RSVD_POWER_ITER)

S_ell_deim_svd = solve_elliptic_path_pod_deim_rom(
    ell_path["params"], V_ell_svd, U_ell_f_svd, ell_config, ell_path["L"], rhs_ell
)
S_ell_deim_rsvd = solve_elliptic_path_pod_deim_rom(
    ell_path["params"], V_ell_rsvd, U_ell_f_rsvd, ell_config, ell_path["L"], rhs_ell
)

err_ell_deim_svd = relative_fro_error(S_ell_path, S_ell_deim_svd)
err_ell_deim_rsvd = relative_fro_error(S_ell_path, S_ell_deim_rsvd)

print(f"Elliptic POD-DEIM SVD ROM error:  {err_ell_deim_svd:.3e}")
print(f"Elliptic POD-DEIM rSVD ROM error: {err_ell_deim_rsvd:.3e}")

## 10. Generate the four requested GIFs

In [ ]:
gif_1 = build_gif_three_columns(
    S_full=S_ac,
    S_svd=S_ac_pod_svd,
    S_rsvd=S_ac_pod_rsvd,
    n=ac_config.n,
    times_or_labels=times_ac,
    filename="allen_cahn_pod.gif",
    suptitle=f"POD on Allen-Cahn equation, r={R_STATE}",
    left_title="Full-order PDE",
    middle_title="POD-SVD ROM",
    right_title="POD-rSVD ROM",
    middle_error=err_ac_pod_svd,
    right_error=err_ac_pod_rsvd,
)

gif_2 = build_gif_three_columns(
    S_full=S_ac,
    S_svd=S_ac_deim_svd,
    S_rsvd=S_ac_deim_rsvd,
    n=ac_config.n,
    times_or_labels=times_ac,
    filename="allen_cahn_deim.gif",
    suptitle=f"POD-DEIM on Allen-Cahn equation, r={R_STATE}, m={M_DEIM}",
    left_title="Full-order PDE",
    middle_title="POD-DEIM SVD ROM",
    right_title="POD-DEIM rSVD ROM",
    middle_error=err_ac_deim_svd,
    right_error=err_ac_deim_rsvd,
)

gif_3 = build_gif_three_columns(
    S_full=S_ell_path,
    S_svd=S_ell_pod_svd,
    S_rsvd=S_ell_pod_rsvd,
    n=ell_config.n,
    times_or_labels=labels_ell,
    filename="elliptic_pod.gif",
    suptitle=f"POD on nonlinear elliptic problem, r={R_STATE}",
    left_title="Full-order PDE",
    middle_title="POD-SVD ROM",
    right_title="POD-rSVD ROM",
    middle_error=err_ell_pod_svd,
    right_error=err_ell_pod_rsvd,
)

gif_4 = build_gif_three_columns(
    S_full=S_ell_path,
    S_svd=S_ell_deim_svd,
    S_rsvd=S_ell_deim_rsvd,
    n=ell_config.n,
    times_or_labels=labels_ell,
    filename="elliptic_deim.gif",
    suptitle=f"POD-DEIM on nonlinear elliptic problem, r={R_STATE}, m={M_DEIM}",
    left_title="Full-order PDE",
    middle_title="POD-DEIM SVD ROM",
    right_title="POD-DEIM rSVD ROM",
    middle_error=err_ell_deim_svd,
    right_error=err_ell_deim_rsvd,
)

print("Generated GIF files:")
for path in [gif_1, gif_2, gif_3, gif_4]:
    print(" -", path)

## 11. Notes for the presentation / oral explanation

- The Allen--Cahn GIFs show a true time-dependent PDE evolution.
- The elliptic GIFs show a parameter-dependent evolution along a diagonal path in parameter space, because the elliptic problem is steady-state.
- `POD-SVD ROM` uses a reduced state basis built with deterministic SVD.
- `POD-rSVD ROM` uses a reduced state basis built with randomized SVD.
- `POD-DEIM SVD ROM` uses deterministic SVD both for the state basis and for the nonlinear DEIM basis.
- `POD-DEIM rSVD ROM` uses randomized SVD for both the state basis and the nonlinear DEIM basis.
- The error in each reduced column is the relative Frobenius error over all frames:
  \[
  \frac{\|S_{\mathrm{FOM}} - S_{\mathrm{ROM}}\|_F}{\|S_{\mathrm{FOM}}\|_F}.
  \]
- A low rank `r=5` and `m=5` is intentionally used so that the difference between full-order and reduced models is visible.